## 1. 주문 상세 금액 계산

### 목적
주문 상세(order_items)의 수량과 단가를 곱해 항목별 결제 금액을 계산한다.

### 실행 전 예상
order_items에 item_amount라는 새 컬럼이 추가되고, 행 수는 원본과 동일할 것이다.

In [1]:
import pandas as pd
from pathlib import Path

# 노트북 위치 기준으로 프로젝트 루트 찾기
PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"

order_items = pd.read_csv(DATA_DIR / "order_items.csv")
order_items["item_amount"] = order_items["quantity"] * order_items["unit_price"]
order_items.head()

,order_item_id,order_id,product_id,quantity,unit_price,item_amount
0,1,1,216,4,13600,54400
1,2,1,34,1,87000,87000
2,3,1,187,1,93000,93000
3,4,1,112,1,225000,225000
4,5,2,214,1,64000,64000


### 실제 결과
item_amount 컬럼이 정상적으로 추가되었고, head()로 확인한 상위 5개 행에서
quantity * unit_price 값과 item_amount 값이 일치함을 확인했다.

### 검증 방법
order_items 전체 행 수가 원본과 동일한지, 
그리고 임의의 한 행에서 quantity * unit_price 계산값과 item_amount가 같은지 확인했다.

### AI 코드 수정 내용
**AI가 처음 제안한 코드:**

```python
order_items = pd.read_csv("data/raw/order_items.csv")
```

**문제:** 노트북 실행 위치가 notebooks/ 폴더 기준이라 상대경로가 파일을 찾지 못하는 FileNotFoundError 발생

**수정한 코드:**

```python
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"
order_items = pd.read_csv(DATA_DIR / "order_items.csv")
```

**수정 이유:** 노트북이 어떤 위치에서 실행되든 항상 프로젝트 루트 기준으로 데이터 폴더를 찾도록 경로를 고정함

In [2]:
# 검증: item_amount가 실제로 quantity * unit_price와 같은지 확인
sample = order_items.iloc[0]
assert sample["item_amount"] == sample["quantity"] * sample["unit_price"]
print("검증 통과: item_amount 계산이 정확합니다.")

검증 통과: item_amount 계산이 정확합니다.


## 2. 주문, 주문 상세, 상품 데이터 병합

### 목적
orders, order_items, products를 연결해 주문별 상품/카테고리 정보를 포함한 통합 데이터를 만든다.

### 실행 전 예상
병합 후 행 수는 order_items 원본 행 수와 같아야 하고, category 컬럼에 결측치가 없어야 한다.

In [3]:
# orders, products 데이터 불러오기
orders = pd.read_csv(DATA_DIR / "orders.csv")
products = pd.read_csv(DATA_DIR / "products.csv")

# 병합 전 원본 행 수 저장
before_rows = len(order_items)

# 1. order_items + orders 병합 (order_id 기준)
merged = order_items.merge(orders, on="order_id", how="left")

# 2. 위 결과 + products 병합 (product_id 기준)
merged = merged.merge(products, on="product_id", how="left")

# 병합 후 행 수 확인
after_rows = len(merged)

print(f"병합 전 행 수: {before_rows}")
print(f"병합 후 행 수: {after_rows}")
print(f"행 수 일치 여부: {before_rows == after_rows}")

# category 컬럼 결측치 확인
missing_category = merged["category"].isna().sum()
print(f"category 결측치 개수: {missing_category}")

merged.head()

병합 전 행 수: 14603
병합 후 행 수: 14603
행 수 일치 여부: True
category 결측치 개수: 0


,order_item_id,order_id,product_id,quantity,unit_price,item_amount,customer_id,order_date,payment_method,order_status,product_name,category,price
0,1,1,216,4,13600,54400,1121,2026-05-18,간편결제,배송중,컴팩트 에세이 그린 P216,도서,16000
1,2,1,34,1,87000,87000,1121,2026-05-18,간편결제,배송중,클래식 클렌징 폼 화이트 P034,뷰티,87000
2,3,1,187,1,93000,93000,1121,2026-05-18,간편결제,배송중,데일리 러닝 벨트 그레이 P187,스포츠,93000
3,4,1,112,1,225000,225000,1121,2026-05-18,간편결제,배송중,플러스 핸디 청소기 블루 P112,생활가전,250000
4,5,2,214,1,64000,64000,993,2026-02-01,신용카드,환불,스마트 클렌징 폼 화이트 P214,뷰티,64000


### 실제 결과
병합 전 행 수(14603)와 병합 후 행 수(14603)가 일치했고, category 컬럼의 결측치는
0개로 확인되어 모든 행이 정상적으로 매칭되었다. customer_id, order_date, category 등
필요한 컬럼이 모두 정상적으로 추가되었다.

### 검증 방법
병합 전 order_items 행 수와 병합 후 merged 행 수를 비교(before_rows == after_rows)하여
병합으로 인한 행 중복 여부를 확인했다.
또한 category 컬럼의 결측치 개수를 isna().sum()으로 확인하여 product_id 매칭이
누락된 행이 없는지 검증했다.

### AI 코드 수정 내용
AI가 제안한 코드를 수정 없이 그대로 사용했다. 다만 실행 전 orders.csv와
products.csv의 실제 컬럼명(customer_id, order_date, payment_method, order_status,
product_name, category, price)이 AI에게 전달한 정보와 일치하는지 직접 확인한 후 적용했다.

## 3. 핵심 지표 계산

### 목적
쇼핑몰 현황을 요약할 수 있는 핵심 지표(전체 주문 수, 총 주문 수량, 총 주문 금액, 
평균 주문 금액, 주문 상태별 건수)를 계산한다.

### 실행 전 예상
전체 주문 수는 order_id 기준 고유값 개수와 같아야 하고, 
총 주문 금액은 merged의 item_amount 합계와 같아야 한다.
평균 주문 금액은 총 주문 금액을 총 주문 수로 나눈 값과 일치해야 한다.

In [4]:
# 전체 주문 수 (order_id 기준 중복 제거)
total_orders = merged["order_id"].nunique()

# 총 주문 수량
total_quantity = merged["quantity"].sum()

# 총 주문 금액
total_amount = merged["item_amount"].sum()

# 평균 주문 금액 (주문 건당 평균, order_id 기준으로 묶어서 계산)
order_amount = merged.groupby("order_id")["item_amount"].sum()
average_order_amount = order_amount.mean()

# 주문 상태별 건수 (order_id 기준 중복 제거 후 상태별 카운트)
status_counts = merged.drop_duplicates("order_id")["order_status"].value_counts()

# 결과 출력
print(f"전체 주문 수: {total_orders}")
print(f"총 주문 수량: {total_quantity}")
print(f"총 주문 금액: {total_amount:,}원")
print(f"평균 주문 금액: {average_order_amount:,.0f}원")
print("\n주문 상태별 건수:")
print(status_counts)

전체 주문 수: 6000
총 주문 수량: 23596
총 주문 금액: 1,837,774,300원
평균 주문 금액: 306,296원

주문 상태별 건수:
order_status
배송완료    4862
취소       458
환불       352
배송중      163
배송준비      92
결제완료      73
Name: count, dtype: int64


In [5]:
#반복 계산 함수로 만들기
#3. 핵심지표 계산과 겹치는 작업 이 있어서 함수로 정의함
#추후 src/analysis.py에 사용할 예정

def calculate_core_metrics(merged_df):
    """
    병합된 데이터프레임에서 핵심 지표를 계산해 딕셔너리로 반환한다.
    """
    total_orders = merged_df["order_id"].nunique()
    total_quantity = merged_df["quantity"].sum()
    total_amount = merged_df["item_amount"].sum()
    order_amount = merged_df.groupby("order_id")["item_amount"].sum()
    average_order_amount = order_amount.mean()
    status_counts = merged_df.drop_duplicates("order_id")["order_status"].value_counts()

    return {
        "total_orders": total_orders,
        "total_quantity": total_quantity,
        "total_amount": total_amount,
        "average_order_amount": average_order_amount,
        "status_counts": status_counts
    }

# 함수 실행 및 검증
metrics = calculate_core_metrics(merged)
print(metrics)

{'total_orders': 6000, 'total_quantity': np.int64(23596), 'total_amount': np.int64(1837774300), 'average_order_amount': np.float64(306295.7166666667), 'status_counts': order_status
배송완료    4862
취소       458
환불       352
배송중      163
배송준비      92
결제완료      73
Name: count, dtype: int64}


### 실제 결과
전체 주문 수, 총 주문 수량, 총 주문 금액, 평균 주문 금액, 주문 상태별 건수가
모두 정상적으로 계산되었다. 직접 계산한 평균값과 함수 결과가 일치함을 확인했다.

### 검증 방법
- 총 주문 금액이 merged의 item_amount 합계와 일치하는지 assert로 확인했다.
- 평균 주문 금액을 수동으로 (총액 / 주문수) 계산한 값과 함수 결과를 비교했다.
- 두 값 모두 일치했다.

### AI 코드 수정 내용
AI가 제안한 기본 계산 로직에, 재사용을 위해 calculate_core_metrics()라는
함수로 감싸는 구조를 추가했다. 이는 Streamlit 앱(C 담당)에서 동일한 로직을
반복 호출할 수 있도록 하기 위함이다.

## 4. 카테고리별 매출 집계

### 목적
상품 카테고리별 총 매출(item_amount 합계)을 계산해 어떤 카테고리의 매출이 높은지 파악한다.

### 실행 전 예상
category별로 한 행씩 결과가 생성되고, 매출 금액 기준 내림차순으로 정렬될 것이다.
전체 카테고리 매출 합계는 merged의 item_amount 총합과 일치해야 한다.

In [6]:
# 카테고리별 매출 집계 (내림차순 정렬)
category_sales = (
    merged.groupby("category", as_index=False)["item_amount"]
    .sum()
    .sort_values("item_amount", ascending=False)
)

category_sales.columns = ["category", "total_sales"]
category_sales

,category,total_sales
4,생활가전,457423100
8,패션,313067200
7,전자기기,286750100
9,홈인테리어,173982200
5,스포츠,154676700
3,뷰티,135823500
6,식품,116157900
2,반려동물,110816200
1,문구,46005800
0,도서,43071600


### 실제 결과
category별 매출 합계가 정상적으로 계산되었고, 매출 금액 기준 내림차순으로
정렬되었다. 카테고리별 매출 합계의 총합이 전체 item_amount 합계와 일치함을 확인했다.

### 검증 방법
category_sales의 total_sales 합계와 merged의 item_amount 전체 합계를
assert로 비교하여 그룹핑 과정에서 데이터 누락이나 중복이 없는지 검증했다.

### AI 코드 수정 내용
AI가 제안한 groupby 코드를 그대로 사용했으며, 컬럼명을 category, total_sales로
명확하게 변경해 가독성을 높였다.

## 5. 주문 상태별 건수
    핵심지표 계산에 포함 되어 있음

## 6. 월별 주문 금액 계산

### 목적
월별 주문 금액 추이를 계산해 매출이 어떻게 변화하는지 파악한다.

### 실행 전 예상
order_date를 월 단위로 변환한 뒤 집계하면, 연월별로 한 행씩 결과가 생성될 것이다.
전체 월별 매출 합계는 merged의 item_amount 총합과 일치해야 한다.

In [7]:
# order_date를 날짜 타입으로 변환
merged["order_date"] = pd.to_datetime(merged["order_date"])

# 연-월 컬럼 생성
merged["order_month"] = merged["order_date"].dt.to_period("M").astype(str)

# 월별 주문 금액 집계
monthly_sales = (
    merged.groupby("order_month", as_index=False)["item_amount"]
    .sum()
    .sort_values("order_month")
)

monthly_sales.columns = ["order_month", "total_sales"]
monthly_sales

,order_month,total_sales
0,2025-01,87322800
1,2025-02,68504700
2,2025-03,85551000
3,2025-04,104790900
4,2025-05,105969700
5,2025-06,82491800
6,2025-07,86357700
7,2025-08,81312400
8,2025-09,98212700
9,2025-10,84413300


### 실제 결과
월별로 order_month 기준 집계 결과가 정상적으로 생성되었고, 시간순으로 정렬되었다.
전체 월별 매출 합계가 merged의 item_amount 총합과 일치함을 확인했다.

### 검증 방법
monthly_sales의 total_sales 합계와 merged의 item_amount 전체 합계를
assert로 비교하여 월별 그룹핑 과정에서 데이터 누락이 없는지 검증했다.

### AI 코드 수정 내용
AI가 제안한 기본 로직을 그대로 사용했으며, dt.to_period("M")로 생성된 Period 타입을
astype(str)로 변환해 groupby와 정렬이 문자열 기준으로 안정적으로 동작하도록 했다.

## 7. 반복 계산 함수로 정리 (3번에 이어 추가)

### 목적
카테고리별 매출 집계, 월별 매출 집계 로직을 함수로 만들어
Streamlit에서 필터링된 데이터에도 재사용할 수 있도록 한다.

### 실행 전 예상
함수로 감싸도 로직은 동일하므로, 앞서 계산한 category_sales, monthly_sales와
동일한 결과가 나올 것이다. 각 결과의 total_sales 합계는 merged의 item_amount
전체 합계와 일치할 것이다.

In [8]:
def calculate_category_sales(merged_df):
    """카테고리별 매출 합계를 계산해 내림차순으로 반환한다."""
    result = (
        merged_df.groupby("category", as_index=False)["item_amount"]
        .sum()
        .sort_values("item_amount", ascending=False)
    )
    result.columns = ["category", "total_sales"]
    return result


def calculate_monthly_sales(merged_df):
    """월별 주문 금액 합계를 계산해 시간순으로 반환한다."""
    df = merged_df.copy()
    df["order_date"] = pd.to_datetime(df["order_date"])
    df["order_month"] = df["order_date"].dt.to_period("M").astype(str)
    result = (
        df.groupby("order_month", as_index=False)["item_amount"]
        .sum()
        .sort_values("order_month")
    )
    result.columns = ["order_month", "total_sales"]
    return result


# 함수 실행 및 검증
category_result = calculate_category_sales(merged)
monthly_result = calculate_monthly_sales(merged)

assert category_result["total_sales"].sum() == merged["item_amount"].sum()
assert monthly_result["total_sales"].sum() == merged["item_amount"].sum()
print("검증 통과: 함수화된 집계 결과가 기존 결과와 일치합니다.")

검증 통과: 함수화된 집계 결과가 기존 결과와 일치합니다.


### 실제 결과
calculate_category_sales(), calculate_monthly_sales() 함수 실행 결과가
기존에 개별적으로 계산했던 category_sales, monthly_sales와 동일함을 확인했다.

### 검증 방법
함수 실행 결과의 total_sales 합계가 merged의 item_amount 전체 합계와
일치하는지 assert로 검증했다. 두 함수 모두 검증을 통과했다.

### AI 코드 수정 내용
AI가 제안한 기본 함수 구조에, merged_df.copy()를 추가해 함수 내부에서
원본 데이터프레임(merged)이 의도치 않게 수정되는 부작용을 방지하도록 수정했다.
이는 Streamlit에서 필터링된 데이터를 반복적으로 함수에 전달할 때
원본 데이터가 오염되지 않도록 하기 위함이다.

In [9]:
import sys
sys.path.append(str(PROJECT_ROOT))
from src.analysis import load_merged_data, calculate_core_metrics, calculate_category_sales, calculate_monthly_sales

# src/analysis.py의 함수들이 노트북 결과와 동일한지 최종 확인
test_merged = load_merged_data(DATA_DIR)
test_metrics = calculate_core_metrics(test_merged)

assert test_metrics["total_amount"] == merged["item_amount"].sum()
print("검증 통과: src/analysis.py 함수가 노트북 결과와 일치합니다.")

검증 통과: src/analysis.py 함수가 노트북 결과와 일치합니다.
